# Phase 0/1 exploration
1. CUDA check + toy training loop (Phase 0)
2. Look at raw vs filtered wrist EDA, labels, and windows (Phase 1)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import torch, torch.nn as nn
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Toy loop: learn y = 2x + 1. The 5 core steps: forward -> loss -> zero_grad -> backward -> step
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
x = torch.randn(256, 1, device=dev); y = 2 * x + 1
model = nn.Linear(1, 1).to(dev)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
for step in range(200):
    loss = nn.functional.mse_loss(model(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0: print(step, loss.item())
print('w, b =', model.weight.item(), model.bias.item())

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.utils import DATA_PROC, load_config
from src.data.preprocess import lowpass, zscore, load_subject
cfg = load_config()['data']
z = np.load(DATA_PROC / 'S2.npz')
eda, lab = z['eda'], z['labels']
t = np.arange(len(eda)) / cfg['fs'] / 60
filt = zscore(lowpass(eda.astype(float), cfg['fs'], cfg['lowpass_hz'], cfg['lowpass_order']))
fig, ax = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
ax[0].plot(t, eda); ax[0].set_ylabel('raw EDA (uS)')
ax[1].plot(t, filt); ax[1].set_ylabel('filtered, z-scored')
ax[2].plot(t, lab); ax[2].set_ylabel('label'); ax[2].set_xlabel('minutes')
plt.show()

In [ ]:
for s in load_config()['subjects']:
    X, y, _ = load_subject(s)
    print(s, X.shape, 'stress', int(y.sum()), 'non-stress', int((1 - y).sum()))